This notebook tests the FSRCNN model using a **given ground-truth image** by creating its low-resolution
counterpart and applying super-resolution on it.

In [ ]:
from pathlib import Path
import os

import numpy as np
import torch
from PIL import Image
import onnxruntime as ort
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

Adjust this part based on current project structure.

In [ ]:
project_root = Path(os.getcwd()).parent
onnx_file = r""
onnx_file_path = project_root / "models" / onnx_file

Inference configuration:
- Scale factor: How downscaled the LR input is expected to be. Should be the same as the one used during
the model training.
- Number of channels: The number of colour channels of the image. 3 for RGB, 1 for greyscale.
- Input height and width: The dimensions of the downscaled LR
- Device: CPU or GPU based on avaialble hardware. Value assigned automatically.

In [ ]:
scale_factor = 4
num_channels = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Fetch all available providers and use GPU or CPU providers accordingly.

In [ ]:
# get available providers and use CUDA if available, otherwise use CPU
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if device.type == "cuda" else ["CPUExecutionProvider"]
available_providers = ort.get_available_providers()
providers = [provider for provider in providers if provider in available_providers]

# create inference session
session = ort.InferenceSession(onnx_file_path, providers=providers)

Input configuration. Expected ground truth image:
- square, 3:4, or 5:4 aspect ratios or similar
- colour image (RGB)

In [ ]:
ground_truth_path = r''

# Load given image
ground_truth_image = Image.open(ground_truth_path).convert("RGB")

In [ ]:
# Resize to match the expected input size
#  i.e. to be divisible by the scale factor
ground_truth_image = Image.open(ground_truth_path).convert("RGB")
gt_adjusted_width = (ground_truth_image.width // scale_factor) * scale_factor
gt_adjusted_height = (ground_truth_image.height // scale_factor) * scale_factor
ground_truth_image = ground_truth_image.resize((gt_adjusted_width, gt_adjusted_height))

In [ ]:
lr_image = ground_truth_image.resize((gt_adjusted_width // scale_factor, gt_adjusted_height // scale_factor),
                           resample=Image.NEAREST)

The LR input (and ground truth if given) are converted into tensors (and normalized to \[0,1\] as the
`transforms.ToTensor()` function facilitates this.

In [ ]:
# Function that transforms from Pillow RGB to PyTorch tensor and normalizes to [0, 1]
transform_to_tensor = transforms.Compose([
    transforms.ToTensor()
])

# Use PyTorch transforms to convert to tensor and normalize
ground_truth_tensor = transform_to_tensor(ground_truth_image)
lr_tensor = transform_to_tensor(lr_image)

Then the tensors are converted into numpy arrays, and the input becomes a numpy array of the form `(B, C, H,
 W)` where:
 - `B` is the batch dimension (this represents the number of images in the batch, in this case 1)
- `C` is the number of colour channels (3 for RGB)
- `H` and `W` are the height and width of the image

Then, the model runs with `session.run`, returning a numpy array image.

In [ ]:
ground_truth_numpy = ground_truth_tensor.numpy()
# The ONNX runtime expects a numpy array
lr_numpy = lr_tensor.numpy()
# Batch dimension is expected (B, C, H, W)
lr_numpy_with_batch = np.expand_dims(lr_numpy, axis=0)

sr_numpy = session.run(output_names=["sr"], input_feed={"lr": lr_numpy_with_batch})[0]

The SR output is converted to an image to be used for visualization, normalized and transposed to `(H, W,
C)` format.

In [ ]:
transform_to_PIL = transforms.ToPILImage()

# Convert to PIL image
# Clip to [0, 1] values and tranpose to (H, W, C)
sr_numpy = sr_numpy[0]
sr_hwc = np.clip(sr_numpy.transpose(1, 2, 0), 0.0, 1.0)
sr_image = Image.fromarray((sr_hwc * 255.0).round().astype(np.uint8))

In [ ]:
# Visualize result
num_cols = 2

# Include ground truth image if provided
if ground_truth_image:
    num_cols = 3

fig, axes = plt.subplots(1, num_cols, figsize=(15, 5))

axes[0].imshow(lr_image, interpolation="none")
axes[0].set_title("LR Image")
axes[0].axis("off")

axes[1].imshow(sr_image)
axes[1].set_title("SR Image")
axes[1].axis("off")

axes[2].imshow(ground_truth_image)
axes[2].set_title("Ground Truth")
axes[2].axis("off")

plt.tight_layout()
plt.show()

A visualization of the absolute pixel difference between the ground truth and the SR output is also
available. This is to check how much the SR output differs from the actual ground truth.

In [ ]:
# Compute absolute difference
diff = np.abs(sr_numpy.astype(float) - ground_truth_numpy.astype(float))
# Transpose to (H, W, C) format
diff = diff.transpose(1, 2, 0)

# Compute mean difference (across all channels)
mean_diff = np.mean(diff, axis=2)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(mean_diff, cmap="hot", interpolation="none")
ax.set_title("Ground Truth vs SR Output Pixel Difference")
fig.colorbar(im, ax=ax)
plt.show()